# BÁO CÁO NGHIÊN CỨU VÀ THỰC NGHIỆM KỸ THUẬT RAG + LLAMAINDEEX
## Đề bài 2: Hệ thống Tra cứu Luật Lao động & Hợp đồng Nhân sự
**Dự án Học thuật EduNext | 5 Task kỹ thuật bắt buộc**

---

### CELL 1: Cài đặt và Nạp các thư viện bắt buộc (100% Free Stack)

In [1]:
import os
import sys
from pathlib import Path

# Thêm đường dẫn gốc Backend vào sys.path để import các services
backend_dir = Path.cwd().parent
if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

from llama_index.core import Document, VectorStoreIndex, Settings as LlamaSettings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

print("✅ Nạp các thư viện LlamaIndex, HuggingFace, ChromaDB thành công!")

ModuleNotFoundError: No module named 'llama_index'

### CELL 2: Task 1 – Dense Vector Indexing & Top-K Similarity Search
- Nạp các passages về Thời hạn báo trước, Chế độ thai sản, Lương làm thêm giờ.
- Tạo Vector Index với HuggingFace Local Embedding + ChromaDB.
- Đo Top-K Similarity Score.

In [ ]:
from app.services.indexing_service import indexing_service

# Khởi tạo hoặc xóa re-index lại từ đầu
index = indexing_service.get_vector_store_index(reindex=True)
print("✅ Task 1: Tạo Dense Vector Index thành công với LlamaIndex & ChromaDB!")

# Thử nghiệm Dense Vector Search Top-K
query_test = "Người lao động hợp đồng 24 tháng muốn nghỉ việc thì phải báo trước bao nhiêu ngày?"
results, latency = indexing_service.dense_similarity_search(query=query_test, top_k=3)

print(f"\n🔎 Query: '{query_test}'")
print(f"⏱️ Latency: {latency} ms")
for i, res in enumerate(results, 1):
    print(f"\n--- Top {i} [Score: {res.score}] ---")
    print(f"File: {res.file_name} | Tiêu đề: {res.title}")
    print(f"Nội dung: {res.text[:180]}...")

### CELL 3: Task 2 – Retrieve-then-Generate (RAG vs Non-RAG Comparison)
- So sánh chất lượng câu trả lời khi dùng RAG vs khi không dùng RAG (Non-RAG).
- Làm nổi bật hiện tượng Hallucination của LLM.

In [ ]:
from app.services.rag_service import rag_service

question = "Lao động ký hợp đồng 24 tháng muốn nghỉ việc thì phải báo trước bao nhiêu ngày?"

# 1. Thực thi Chế độ RAG
rag_res = rag_service.execute_rag(question=question, use_rag=True)

# 2. Thực thi Chế độ Non-RAG
non_rag_res = rag_service.execute_rag(question=question, use_rag=False)

print("="*60)
print("🟢 BẢNG SO SÁNH TASK 2: RAG VS NON-RAG")
print("="*60)
print(f"📌 Câu hỏi: {question}\n")
print(f"🟢 RAG ANSWER (Có truy xuất ngữ cảnh):\n{rag_res.answer}\n")
print(f"🔴 NON-RAG ANSWER (Không có ngữ cảnh - Hallucination):\n{non_rag_res.answer}\n")
print(f"⏱️ RAG Execution Time: {rag_res.execution_time_seconds} giây")

### CELL 4: Task 3 – Semantic Chunking & Benchmark Chunk Size
- Phân đoạn tài liệu dài theo các chunk size: 128, 256, 1024, Unchunked.
- Đo Retrieval Precision, latency và số lượng chunks tạo ra.

In [ ]:
from app.services.chunking_service import chunking_service

chunk_res = chunking_service.compare_chunk_sizes()

print("="*70)
print(f"📊 KẾT QUẢ THỰC NGHIỆM TASK 3: SEMANTIC CHUNKING BENCHMARK")
print(f"File: {chunk_res.document_name}")
print(f"Query: {chunk_res.query}")
print("="*70)
print(f"{'Chunk Size':<12} | {'Chunks':<8} | {'Avg Len':<10} | {'Latency (ms)':<14} | {'Top Score':<10} | {'Precision':<10}")
print("-"*70)
for r in chunk_res.results:
    sz_str = str(r.chunk_size) if r.chunk_size > 0 else "Unchunked"
    print(f"{sz_str:<12} | {r.total_chunks:<8} | {r.avg_chunk_char_length:<10} | {r.retrieval_time_ms:<14} | {r.top_score:<10} | {r.precision_percent}%")

### CELL 5: Task 4 – Metadata Filtering Pre-Retrieval
- Gắn metadata có cấu trúc (`loai_hop_dong`, `chu_de`, `phap_ly`, `doi_tuong`).
- Áp dụng pre-filtering trước khi thực hiện RAG vector retrieval.

In [ ]:
# Tra cứu với Metadata Filter: Chỉ lọc bài viết thuộc chủ đề 'tien_luong_thuong'
question_m = "Lương làm ngày lễ 30/4 tính bao nhiêu %?"
filter_meta = {"chu_de": "tien_luong_thuong"}

filtered_res = rag_service.execute_rag(
    question=question_m,
    use_rag=True,
    filters_dict=filter_meta
)

print("="*60)
print("🎯 KẾT QUẢ TASK 4: METADATA FILTERING PRE-RETRIEVAL")
print("="*60)
print(f"Bộ lọc áp dụng: {filter_meta}")
print(f"Số lượng Nguồn thỏa mãn bộ lọc: {len(filtered_res.sources)}")
for s in filtered_res.sources:
    print(f"- Node ID: {s.node_id} | Title: {s.title} | Metadata: {s.metadata}")
print(f"\n💬 Trả lời RAG: {filtered_res.answer}")

### CELL 6: Task 5 – So sánh 2 Embedding Models
- So sánh `sentence-transformers/all-MiniLM-L6-v2` vs `BAAI/bge-small-en-v1.5`.
- Đo thời gian Indexing, Latency, Similarity Score và Vector Dimension.

In [ ]:
from app.services.embedding_service import embedding_service

embed_comp = embedding_service.compare_embedding_models()

print("="*75)
print("⚖️ BẢNG SO SÁNH TASK 5: EMBEDDING MODELS BENCHMARK")
print("="*75)
print(f"{'Model Name':<38} | {'Dims':<6} | {'Idx Time(ms)':<12} | {'Retrieval(ms)':<14} | {'Top Score':<10}")
print("-"*75)
for r in embed_comp.results:
    print(f"{r.model_name:<38} | {r.dimension:<6} | {r.indexing_time_ms:<12} | {r.retrieval_time_ms:<14} | {r.top_score:<10}")

print(f"\n💡 ĐÁNH GIÁ CHUYÊN MÔN:\n{embed_comp.recommendation}")